In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES
import numpy as np
import math

import matplotlib.pyplot as plt
matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# import xarray as xr
# import uxarray as ux

import os

In [ ]:
#Importing DirectoryManager Class

sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Unstructured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import UnstructuredModelData_Class

In [ ]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = UnstructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

[dataVariables, dataVariables_diag] = ModelData.GetVariableNames()
# dataVariables_diag

In [ ]:
###############
#FUNCTIONS

In [ ]:
def LatLonBoundingBox_Center(campaign="TRACER"):
    if campaign == "TRACER":
        (latCenter, lonCenter) = 29.67, -95.059
    return latCenter,lonCenter

def LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500):
    # Earth radius in km
    R = 6371.0

    # Convert degrees to radians
    latRadians = math.radians(latCenter)

    # Calculate degree offsets
    dLat = (radius_km / R) * (180.0 / math.pi)
    dLon = (radius_km / (R * math.cos(latRadians))) * (180.0 / math.pi)

    # Bounding box
    latMin = latCenter - dLat
    latMax = latCenter + dLat
    lonMin = lonCenter - dLon
    lonMax = lonCenter + dLon

    latBounds = (latMin, latMax)
    lonBounds = (lonMin, lonMax)
    return latBounds, lonBounds

def LatLonBoundingBox_Subset(variable, latBounds, lonBounds):
    """
    Subset a uxarray DataArray to a lat/lon bounding box and return
    the subset and its coordinate arrays (lat/lon arrays match subset size).
    """
    # Perform the subset
    variableSubset = variable.subset.bounding_box(
        lat_bounds=latBounds, lon_bounds=lonBounds
    )

    grid = variableSubset.uxgrid

    # Match coordinates to the subset dimension
    if "n_face" in variableSubset.dims:
        indices = variableSubset["n_face"].values
        lat = grid.face_lat.values[indices]
        lon = grid.face_lon.values[indices]
        grid_type = "face"

    elif "n_edge" in variableSubset.dims:
        indices = variableSubset["n_edge"].values
        lat = grid.edge_lat.values[indices]
        lon = grid.edge_lon.values[indices]
        grid_type = "edge"

    elif "n_node" in variableSubset.dims:
        indices = variableSubset["n_node"].values
        lat = grid.node_lat.values[indices]
        lon = grid.node_lon.values[indices]
        grid_type = "node"

    else:
        raise ValueError(f"Could not determine grid type from dims: {variable.dims}")

    return variableSubset, lat, lon

In [ ]:
def GetVariable(varName):
    if varName in dataVariables:
        return data[varName].isel(Time=0)
    elif varName in dataVariables_diag:
        return data_diag[varName].isel(Time=0)
    elif varName == "greenfrac":
        return ModelData.staticData["greenfrac"].isel(nMonths=6)

def GetVariable_Subset(varName):  
    variable = GetVariable(varName)
    [latCenter,lonCenter] = LatLonBoundingBox_Center(campaign="TRACER")
    [latBounds, lonBounds] = LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=1000)
    variableSubset, lat, lon = LatLonBoundingBox_Subset(variable,latBounds, lonBounds)
    return variableSubset, lat, lon

In [ ]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(variable, varName, lat,lon, outputFile, save=False, cmap="viridis", title=""):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    # Create figure
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(9, 5)
    )

    vals = variable.values.squeeze()
    # Scatter/contour fill (tricontourf works for unstructured grids)
    im = ax.tricontourf(
        lon, lat, vals,
        levels=60,
        cmap=cmap,
        transform=ccrs.PlateCarree(),
    )

    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar + title
    plt.colorbar(im, ax=ax, orientation="vertical", label=varName)
    ax.set_title(title or varName)

    # Save or display
    if save:
        plt.savefig(outputFile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved static PNG: {outputFile}")
        return None
    else:
        plt.show()
        return fig


In [ ]:
def BuildVariableDictionary(varNames):
    variableDictionary = {}
    for varName in varNames:
        print(f"Adding {varName}")

        # Getting Output Directory
        folderName = varName
        timeString = ModelData.timeStrings[t]
        fileName = f"{varName}_{timeString}.png"
        outputFile = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
        
        # Handle addition of two variables
        if '+' in varName:
            var1, var2 = varName.split('+')
            var1 = var1.strip()
            var2 = var2.strip()
            
            subset1, lat1, lon1 = GetVariable_Subset(var1)
            subset2, lat2, lon2 = GetVariable_Subset(var2)
    
            # Make sure lat/lon are compatible (e.g., same shape)
            if not (np.array_equal(lat1, lat2) and np.array_equal(lon1, lon2)):
                raise ValueError(f"Lat/lon mismatch for {var1} and {var2}")
    
            variableSubset = subset1 + subset2
            lat, lon = lat1, lon1
        else:
            variableSubset, lat, lon = GetVariable_Subset(varName)
    
        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset,
            "lat": lat,
            "lon": lon,
            "outputFile": outputFile
        }
        
    return variableDictionary

def MakePlots(variableDictionary):
    for varName, contents in variableDictionary.items():
        # print(f"Plotting {varName}")
    
        data = contents["data"]
        lat  = contents["lat"]
        lon  = contents["lon"]
        outputFile = contents["outputFile"]
        
        PlotVariable_with_Borders(data, varName, lat, lon, outputFile, save=True, cmap="viridis")

In [ ]:
#################
#RUNNING

In [ ]:
#time loop
num_times = ModelData.NTime
for t in range(num_times):
    if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
    #loading data
    data = ModelData.GetDataTimestep(t)
    data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")

    #defining variable names
    varNames = [
        "surface_pressure",
        "u10", "v10", "q2",
        "hfx", "qfx", "lh",
        "rainnc+rainc",
        "refl10cm_1km"
    ] + (["greenfrac"] if t == 0 else [])
    varNames = ["surface_pressure","u10"]
    # running
    variableDictionary = BuildVariableDictionary(varNames)
    MakePlots(variableDictionary)